# **1. CZYSZCZENIE DANYCH**
Dane zostały przygotowane do analizy poprzez usunięcie niepoprawnych oraz niekompletnych rekordów. Usunięto transakcje bez identyfikatora klienta, anulowane zamówienia oraz rekordy z niepoprawnymi wartościami liczbowymi (Quantity ≤ 0, UnitPrice ≤ 0). Dodatkowo ujednolicono typy danych, usunięto duplikaty oraz dodano kolumnę Revenue, która reprezentuje wartość sprzedaży.

In [3]:
import pandas as pd

df = pd.read_csv("Online_Retail.csv", encoding='ISO-8859-1')

# usunięcie brakujących CustomerID
df = df.dropna(subset=["CustomerID"])

# usunięcie anulowanych faktur (InvoiceNo zaczyna się od "C")
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# usunięcie rekordów z Quantity <= 0
df = df[df["Quantity"] > 0]

# usunięcie rekordów z UnitPrice <= 0
df = df[df["UnitPrice"] > 0]

# konwersja daty
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# usunięcie duplikatów
df = df.drop_duplicates()

# dodanie kolumny Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

/tmp/ipykernel_9335/731393689.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


Proces czyszczenia danych jest kluczowy, ponieważ błędne dane prowadzą do niepoprawnych analiz. Poprawne przygotowanie danych zapewnia wiarygodność wyników i poprawność działania hurtowni danych („garbage in = garbage out”).

# **2. Ziarno (grain)**
Ziarno tabeli faktów zostało określone na poziomie pojedynczej pozycji faktury (InvoiceNo + StockCode). Oznacza to, że jeden wiersz w tabeli faktów reprezentuje sprzedaż konkretnego produktu w ramach danej faktury.

Wybrano takie ziarno, ponieważ zapewnia najwyższy poziom szczegółowości danych, co umożliwia elastyczne analizy oraz agregację wyników do poziomu klienta, dnia lub całej faktury.

Przykładowa analiza możliwa dzięki temu ziarnu to identyfikacja najlepiej sprzedających się produktów w określonym czasie.

# **3. Model Gwiazdy**

Zaprojektowano schemat gwiazdy składający się z jednej tabeli faktów oraz trzech tabel wymiarów. Tabela faktów przechowuje miary sprzedaży, natomiast tabele wymiarów dostarczają kontekstu analitycznego.

## **Tabela faktów: FactSales**

Dane sprzedażowe na poziomie pojedynczej pozycji faktury

**Klucze obce:**

* customer_key

* product_key

* date_key

**Miary:**

* Quantity

* Revenue

##**Wymiar: DimCustomer**

**Klucz sztuczny:** customer_key

**Atrybuty:**

* CustomerID (klucz naturalny)

* Country

##**Wymiar: DimProduct**

**Klucz sztuczny:** product_key

**Atrybuty:**

* StockCode (klucz naturalny)

* Description

##**Wymiar: DimDate**

**Klucz sztuczny:** date_key

**Atrybuty:**

* InvoiceDate
* Year
* Month
* Day








Taki model umożliwia efektywną analizę danych sprzedażowych w różnych przekrojach, takich jak klient, produkt oraz czas.

# **4. Klucze**

W modelu zastosowano zarówno klucze naturalne, jak i klucze sztuczne (surrogate keys).

**Klucze naturalne**

Klucze naturalne pochodzą bezpośrednio z danych źródłowych i identyfikują rekordy w sposób biznesowy:

* CustomerID – identyfikator klienta

* StockCode – identyfikator produktu

* InvoiceDate – data transakcji

**Klucze sztuczne (surrogate keys)**

W każdym wymiarze wprowadzono klucze sztuczne, które są wykorzystywane w tabeli faktów:

* customer_key – w tabeli DimCustomer

* product_key – w tabeli DimProduct

* date_key – w tabeli DimDate

Klucze sztuczne są generowane niezależnie od danych źródłowych i zapewniają stabilność oraz wydajność modelu.

**Zastosowanie w tabeli faktów**

Tabela faktów FactSales wykorzystuje wyłącznie klucze sztuczne jako klucze obce:

* customer_key

* product_key

* date_key

# **5. SCD (Slowly Changing Dimension)**
Zastosowano SCD typu 1 dla wymiaru DimCustomer. Oznacza to, że w przypadku zmiany danych (np. Country), poprzednia wartość jest nadpisywana nową.

Zmiany są wykrywane na podstawie porównania wartości atrybutów dla tego samego CustomerID. W przypadku wykrycia różnicy, rekord jest aktualizowany.

Takie podejście jest proste w implementacji i wystarczające dla podstawowych analiz, jednak nie przechowuje historii zmian.

In [6]:
# DimCustomer (SCD Type 1)

dim_customer = df[["CustomerID", "Country"]].drop_duplicates().copy()
dim_customer["customer_key"] = range(1, len(dim_customer) + 1)

# **6. Implementacja**
Proces implementacji został zrealizowany w języku Python z wykorzystaniem biblioteki Pandas. Obejmuje on wczytanie danych, ich oczyszczenie, a następnie budowę tabel wymiarów oraz tabeli faktów zgodnie z zaprojektowanym schematem gwiazdy.

In [7]:
import pandas as pd

# =========================
# 1. WCZYTANIE DANYCH
# =========================
df = pd.read_csv("Online_Retail.csv", encoding='ISO-8859-1')

# =========================
# 2. CZYSZCZENIE DANYCH
# =========================
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df = df.drop_duplicates()
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# =========================
# 3. WYMIARY
# =========================

# DimCustomer
dim_customer = df[["CustomerID", "Country"]].drop_duplicates().copy()
dim_customer["customer_key"] = range(1, len(dim_customer) + 1)

# DimProduct
dim_product = df[["StockCode", "Description"]].drop_duplicates().copy()
dim_product["product_key"] = range(1, len(dim_product) + 1)

# DimDate
dim_date = df[["InvoiceDate"]].drop_duplicates().copy()
dim_date["date_key"] = range(1, len(dim_date) + 1)
dim_date["Year"] = dim_date["InvoiceDate"].dt.year
dim_date["Month"] = dim_date["InvoiceDate"].dt.month
dim_date["Day"] = dim_date["InvoiceDate"].dt.day

# =========================
# 4. TABELA FAKTÓW
# =========================

fact = df.merge(dim_customer, on=["CustomerID", "Country"])
fact = fact.merge(dim_product, on=["StockCode", "Description"])
fact = fact.merge(dim_date, on="InvoiceDate")

fact_sales = fact[[
    "customer_key",
    "product_key",
    "date_key",
    "Quantity",
    "Revenue"
]]

/tmp/ipykernel_9335/1377625446.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


Poniżej przedstawiono przykładowe rekordy utworzonych tabel wymiarów oraz tabeli faktów.

In [9]:
display(dim_customer.head())
display(dim_product.head())
display(dim_date.head())
display(fact_sales.head())

,CustomerID,Country,customer_key
0,17850.0,United Kingdom,1
9,13047.0,United Kingdom,2
26,12583.0,France,3
46,13748.0,United Kingdom,4
65,15100.0,United Kingdom,5


,StockCode,Description,product_key
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1
1,71053,WHITE METAL LANTERN,2
2,84406B,CREAM CUPID HEARTS COAT HANGER,3
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,4
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,5


,InvoiceDate,date_key,Year,Month,Day
0,2010-12-01 08:26:00,1,2010,12,1
7,2010-12-01 08:28:00,2,2010,12,1
9,2010-12-01 08:34:00,3,2010,12,1
25,2010-12-01 08:35:00,4,2010,12,1
26,2010-12-01 08:45:00,5,2010,12,1


,customer_key,product_key,date_key,Quantity,Revenue
0,1,1,1,6,15.30
1,1,2,1,6,20.34
2,1,3,1,8,22.00
3,1,4,1,6,20.34
4,1,5,1,6,20.34
